# Use Subqueries to Refine Data - BigQuery NY City Bike Public Dataset

## Activity Overview

A subquery is a query that is nested inside of another query. The subquery filters or sorts data to prepare it to be used by the outer query to produce its final result. This allows data professionals to create more nuanced queries that provide specific insights from the data!

For example, perhaps a data analyst working in human resources has been asked to determine the average salary of employees working within a specific department. The analyst can use a subquery to first find the total salary and number of employees within the department. Then, the outer query will use these numbers to calculate the average salary within a department. It's a step-by-step process, each step relying on the one before it.

In this SQL small sandbox, we will use SELECT statements with FROM, WHERE, and GROUP BY clauses to build our subqueries. 

## Objective: 
Create a subquery using SELECT statements with FROM, WHERE, and GROUP BY clauses and analyze the query’s results. 

## Scenario

We work for an organization that is responsible for the safety, efficiency, and maintenance of transportation systems in our city. We have been asked to gather information around the use of Citi Bikes in New York City. This information will be used to convince the mayor and other city officials to invest in a bike sharing and rental system to help push the city toward its environmental and sustainability goals. 

To complete this task, we will create three different subqueries, which will allow us to gather information about the average trip duration by station, compare trip duration by station, and determine the five stations with the longest mean trip durations. 

## Step 1: Acess the Dataset
For this activity, we will need to log in to my BigQuery account to use the public <b>new_york_citibike dataset</b>. Then, use the table called <b>citibike_trips</b>.

In [1]:
from google.cloud import bigquery

print(bigquery.__version__)

3.40.1


In [2]:
client = bigquery.Client()

print(client.project)

myproject001-504709


## Step 2: Create a Subquery to Find the Average Trip Duration By Station
First, find the average trip duration for a particular station. This subquery will be applied to the SELECT statement.

In [3]:
query = """
-- Outer query
SELECT
    subquery.start_station_id,
    subquery.avg_duration
FROM
    ( -- Inner query,determine the average trip duration by station
    SELECT
        start_station_id,
        AVG(tripduration) as avg_duration
FROM bigquery-public-data.new_york_citibike.citibike_trips
GROUP BY start_station_id) as subquery
ORDER BY avg_duration DESC;
"""
df = client.query(query).to_dataframe()
df

,start_station_id,avg_duration
0,3633,71800.000000
1,3040,38351.692308
2,3590,23327.950284
3,3017,22982.666667
4,3649,15286.277563
...,...,...
877,3197,546.558824
878,3215,522.625000
879,3488,375.611111
880,3487,185.666667


In this case there are 882 lines of data that we can use to analyze and compare the average trip durations between each station. We will continue to work with the average trip duration per station in the next query.

## Step 3: Create a Subquery to Compare Trip Duration By Station
We will create a new query to compare the average trip duration per station to the overall average trip duration from all stations. This will provide insights about how long people typically use the bikes that they get from a particular station in comparison to the overall average. \
This subquery will be applied to the FROM clause in a SELECT statement. 

Parameters:
- starttime: a somewhat unique identifier for the results to mark when trips from a particular station started;
- start_station_id: identifies the station ID for each trip;
- tripduration, which measures the length of each trip in seconds.

In [4]:
query = """
SELECT
    starttime,
    start_station_id,
    tripduration,
    
    ( --subquery that will return the average trip duration for each station
        SELECT ROUND(AVG(tripduration),2)
        FROM bigquery-public-data.new_york_citibike.citibike_trips
        WHERE start_station_id = outer_trips.start_station_id
    ) AS avg_duration_for_station,
   
    --subquery that creates the difference_from_avg column,returns the difference between the specific station's average trip duration and the overall average trip duration
    ROUND(tripduration - (
        SELECT AVG(tripduration)
        FROM bigquery-public-data.new_york_citibike.citibike_trips
        WHERE start_station_id = outer_trips.start_station_id), 2) AS difference_from_avg

FROM bigquery-public-data.new_york_citibike.citibike_trips AS outer_trips
ORDER BY difference_from_avg DESC
LIMIT 25;
"""
df = client.query(query).to_dataframe()
df

,starttime,start_station_id,tripduration,avg_duration_for_station,difference_from_avg
0,2018-01-22 18:20:27.512,3082,19510049,2061.72,19507987.28
1,2018-02-21 14:15:10.932,3349,15962256,2908.95,15959347.05
2,2018-03-15 18:21:38.801,3041,15020934,11295.05,15009638.95
3,2018-02-21 15:30:02.388,3042,13931824,14540.53,13917283.47
4,2018-02-12 15:38:54.233,3042,13586276,14540.53,13571735.47
5,2018-03-11 03:52:30.296,3383,12479323,1780.86,12477542.14
6,2018-02-08 21:46:47.029,3064,11749576,2566.28,11747009.72
7,2018-01-28 03:51:28.614,3537,11699746,3812.10,11695933.90
8,2018-01-27 15:37:18.019,501,11138807,1192.56,11137614.44
9,2018-05-05 13:46:13.022,343,10283682,1623.92,10282058.08


In the last column, difference_from_avg, there are some very large differences from the average duration, indicating that these stations have some significant outliers. It would probably be worthwhile to examine that further later on.

## Step 4: Create a Subquerie to Determie the Five Stations With the Longest Mean Trip Duration
Now, we'll compose a new query to filter the data to include only the trips from the five stations with the longest mean trip duration.

The result of the entire query is a list of records from the main table—specifically the tripduration and start_station_id for each record, but only those for records where the start_station_id is among the five stations with the greatest average trip durations. If we examine the query results, we will discover that only five of the start_station_id values are listed in column two. 

In [5]:
query = """
SELECT
    tripduration,
    start_station_id
FROM bigquery-public-data.new_york_citibike.citibike_trips
WHERE start_station_id IN
    ( --inner subquery which creates a derived table called top_five
        SELECT
            start_station_id
        FROM
        (
            SELECT
                start_station_id,
                AVG(tripduration) AS avg_duration
            FROM bigquery-public-data.new_york_citibike.citibike_trips
            GROUP BY start_station_id
        ) AS top_five
        ORDER BY avg_duration DESC
        LIMIT 5
    );
"""
df = client.query(query).to_dataframe()
df

,tripduration,start_station_id
0,5192,3649
1,615,3649
2,1107,3649
3,646,3649
4,509,3649
...,...,...
1763,179,3590
1764,281,3590
1765,2463,3590
1766,415,3590
